# Part E — Hyperparameter Optimisation: Round 2

**Standalone Google Colab notebook.** Run from top to bottom in a fresh runtime. It reads Part C train/validation CSV files and **existing Part E Round 1 results** from Google Drive. It does not require Round 1 notebook variables, does not retrain Random Forest, and does not access the test set.

**Prerequisite:** Run Round 1 first so `Part_E_Results/best_hyperparameters.csv` and `Part_E_Results/optimised_metrics.csv` exist. All Round 2 files are saved separately under `Part_E_Results/Round_2/`.

Searches: Logistic Regression (L2/L1/Elastic Net) and Histogram Gradient Boosting (targeted complexity and regularisation configurations). Primary CV metric: Average Precision; 3-fold stratified CV. Validation metrics use threshold 0.50.

## 1. Load existing data and results


In [1]:
# Part E, Round 2 — 1. Reload original train/validation data and first-round outputs.
# This can run even if the first-round search objects are no longer in memory.
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, fbeta_score, roc_auc_score,
                             average_precision_score, confusion_matrix)

SEED = 42
CV_FOLDS = 3
# Standalone Drive setup: no dependency on the first-round notebook's variables.
import os
if Path('/content').exists():
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        try:
            drive.mount('/content/drive')
        except ValueError as exc:
            raise RuntimeError(
                'Google Drive mount failed. In Colab, reconnect the runtime and mount Drive '
                'manually before rerunning this cell.'
            ) from exc

root_candidates = [
    Path('/content/drive/MyDrive/ML_Group'),
    Path('/content/drive/MyDrive/ML_Group_Assignment'),
]
ROOT = next((p for p in root_candidates if (p / 'Part_E_Results' / 'best_hyperparameters.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError(
        'Could not find Round 1 results in ML_Group or ML_Group_Assignment. '
        'Expected Part_E_Results/best_hyperparameters.csv; run Round 1 first.'
    )
OUT = ROOT / 'Part_E_Results'
ROUND2 = OUT / 'Round_2'

# Follow the same root and processed CSVs as the Part D notebook.
# Prefer CSVs in the project root; otherwise find exactly one matching pair.
if (ROOT / 'train_dataset.csv').is_file() and (ROOT / 'validation_dataset.csv').is_file():
    train_path = ROOT / 'train_dataset.csv'
    val_path = ROOT / 'validation_dataset.csv'
else:
    candidates = [p.parent for p in ROOT.rglob('train_dataset.csv')
                  if (p.parent / 'validation_dataset.csv').is_file()]
    if len(candidates) != 1:
        raise FileNotFoundError(
            f'Expected one matching train/validation CSV folder under {ROOT}; found {len(candidates)}.'
        )
    train_path = candidates[0] / 'train_dataset.csv'
    val_path = candidates[0] / 'validation_dataset.csv'

stage1_best_path = OUT / 'best_hyperparameters.csv'
stage1_metrics_path = OUT / 'optimised_metrics.csv'
for p in (train_path, val_path, stage1_best_path, stage1_metrics_path):
    if not p.is_file():
        raise FileNotFoundError(f'Missing required Part E input: {p}')

ROUND2.mkdir(parents=True, exist_ok=True)

train_df_r2 = pd.read_csv(train_path)
val_df_r2 = pd.read_csv(val_path)
assert train_df_r2.columns.equals(val_df_r2.columns), 'Training/validation columns differ.'
assert 'target' in train_df_r2.columns, 'Missing target column.'
features_r2 = train_df_r2.columns.drop('target')
X_train_r2, y_train_r2 = train_df_r2[features_r2], train_df_r2['target']
X_val_r2, y_val_r2 = val_df_r2[features_r2], val_df_r2['target']
assert not X_train_r2.isnull().any().any(), 'NaN values in Part D training features.'
assert not X_val_r2.isnull().any().any(), 'NaN values in Part D validation features.'

classes_r2 = np.array([0, 1])
weights_r2 = compute_class_weight('balanced', classes=classes_r2, y=y_train_r2)
class_weights_r2 = {int(c): float(w) for c, w in zip(classes_r2, weights_r2)}
cv_r2 = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
stage1_best_r2 = pd.read_csv(stage1_best_path).set_index('Model')
stage1_metrics_r2 = pd.read_csv(stage1_metrics_path).set_index('Model')
for name in ['Logistic Regression', 'Histogram Gradient Boosting']:
    assert name in stage1_best_r2.index and name in stage1_metrics_r2.index

print('Part E Round 2: training', X_train_r2.shape, '| validation', X_val_r2.shape)
print('Same class-weight policy:', class_weights_r2)
print('Stage 1 CV scoring: average_precision | CV folds:', CV_FOLDS)
print('Round 2 outputs:', ROUND2)


Mounted at /content/drive
Part E Round 2: training (198221, 66) | validation (45302, 66)
Same class-weight policy: {0: 0.591888227987196, 1: 3.220696714652455}
Stage 1 CV scoring: average_precision | CV folds: 3
Round 2 outputs: /content/drive/MyDrive/ML_Group/Part_E_Results/Round_2


## 2. Define second-round searches


In [2]:
# Part E, Round 2 — 2. Define targeted, finite search spaces.
# Logistic Regression: local C refinement plus L1/elastic-net feature shrinkage.
# HGB: stronger L2, smaller trees, altered leaf sizes, learning-rate/iteration pairing.

lr_r2_estimator = LogisticRegression(
    C=1.0, solver='lbfgs', max_iter=1000,
    class_weight=class_weights_r2, random_state=SEED
)

# All configurations retain the same class weights and max_iter as Part D.
# The exact first-round winner (L2, lbfgs, C=0.1) is included.
lr_r2_grid = [
    {'solver': ['lbfgs'], 'penalty': ['l2'],
     'C': [0.03, 0.05, 0.10, 0.20, 0.30, 1.0]},
    {'solver': ['saga'], 'penalty': ['l1'],
     'C': [0.03, 0.10, 0.30]},
    {'solver': ['saga'], 'penalty': ['elasticnet'],
     'l1_ratio': [0.5], 'C': [0.03, 0.10, 0.30]},
]

hgb_r2_estimator = HistGradientBoostingClassifier(
    max_iter=100, learning_rate=0.1, max_leaf_nodes=31,
    class_weight=class_weights_r2, random_state=SEED
)

# One candidate = one complete configuration. Never search all combinations blindly.
hgb_reference = {
    'learning_rate': 0.05, 'max_iter': 180,
    'max_leaf_nodes': 15, 'l2_regularization': 10.0,
    'min_samples_leaf': 20,  # sklearn default; included for clarity.
}

hgb_candidates_r2 = [hgb_reference.copy()]
for reg in [30.0, 100.0]:
    hgb_candidates_r2.append({**hgb_reference, 'l2_regularization': reg})
for nodes in [7, 23, 31]:
    hgb_candidates_r2.append({**hgb_reference, 'max_leaf_nodes': nodes})
for leaf in [10, 50, 100]:
    hgb_candidates_r2.append({**hgb_reference, 'min_samples_leaf': leaf})
for rate, iters in [(0.03, 300), (0.05, 300), (0.07, 180), (0.10, 100)]:
    hgb_candidates_r2.append({**hgb_reference, 'learning_rate': rate,
                              'max_iter': iters})
hgb_candidates_r2 += [
    {**hgb_reference, 'l2_regularization': 30.0, 'max_leaf_nodes': 7},
    {**hgb_reference, 'l2_regularization': 30.0, 'min_samples_leaf': 50},
]

# GridSearchCV accepts a list of dictionaries of singleton lists.
hgb_r2_grid = [{k: [v] for k, v in config.items()}
               for config in hgb_candidates_r2]

scoring_r2 = {
    'average_precision': 'average_precision',
    'roc_auc': 'roc_auc',
    'f1': 'f1',
    'precision': 'precision',
    'recall': 'recall',
}

searches_r2 = {
    'Logistic Regression': GridSearchCV(
        lr_r2_estimator, param_grid=lr_r2_grid, cv=cv_r2,
        scoring=scoring_r2, refit='average_precision',
        n_jobs=1, return_train_score=False, error_score='raise', verbose=1
    ),
    'Histogram Gradient Boosting': GridSearchCV(
        hgb_r2_estimator, param_grid=hgb_r2_grid, cv=cv_r2,
        scoring=scoring_r2, refit='average_precision',
        n_jobs=1, return_train_score=False, error_score='raise', verbose=1
    ),
}

print('Logistic Regression:', 12, 'candidates × 3 folds = 36 CV fits')
print('Histogram Gradient Boosting:', len(hgb_r2_grid),
      'candidates × 3 folds =', len(hgb_r2_grid)*CV_FOLDS, 'CV fits')
print('The Round 1 selected configurations are present in the new searches.')


Logistic Regression: 12 candidates × 3 folds = 36 CV fits
Histogram Gradient Boosting: 15 candidates × 3 folds = 45 CV fits
The Round 1 selected configurations are present in the new searches.


## 3. Fit second-round searches


In [3]:
# Part E, Round 2 — 3. Run the targeted CV searches and save their trials.
# This is the only expensive cell. Searches use training data exclusively.
from sklearn.exceptions import ConvergenceWarning

round2_searches_fitted = {}
round2_best_models = {}
round2_records = []

for model_name, search in searches_r2.items():
    print('\n' + '=' * 64 + '\nROUND 2:', model_name)
    t0 = time.perf_counter()
    with warnings.catch_warnings(record=True) as observed_warnings:
        warnings.simplefilter('always', ConvergenceWarning)
        search.fit(X_train_r2, y_train_r2)
    elapsed = time.perf_counter() - t0

    round2_searches_fitted[model_name] = search
    round2_best_models[model_name] = search.best_estimator_
    trials = pd.DataFrame(search.cv_results_)
    fname = ('logistic_regression' if model_name == 'Logistic Regression'
             else 'histogram_gradient_boosting')
    trials.to_csv(ROUND2 / f'{fname}_cv_trials.csv', index=False)
    best_idx = search.best_index_
    old_cv = float(stage1_best_r2.loc[model_name, 'Best CV Average Precision'])
    new_cv = float(search.best_score_)
    non_convergence = sum(issubclass(w.category, ConvergenceWarning)
                          for w in observed_warnings)

    row = {
        'Model': model_name,
        'Round 1 CV Average Precision': old_cv,
        'Round 2 CV Average Precision': new_cv,
        'CV AP difference': new_cv - old_cv,
        'Round 2 CV AP std': float(trials.loc[best_idx, 'std_test_average_precision']),
        'Round 2 CV ROC-AUC': float(trials.loc[best_idx, 'mean_test_roc_auc']),
        'Round 2 CV F1 @0.50': float(trials.loc[best_idx, 'mean_test_f1']),
        'CV candidates': len(trials),
        'Search time (s)': elapsed,
        'Refit time (s)': float(search.refit_time_),
        'Convergence warnings': non_convergence,
        'Best parameters': json.dumps(search.best_params_, sort_keys=True),
    }
    round2_records.append(row)
    pd.DataFrame(round2_records).to_csv(ROUND2 / 'round2_best_parameters.csv', index=False)
    print('Round 1 mean CV AP:', round(old_cv, 5))
    print('Round 2 mean CV AP:', round(new_cv, 5))
    print('Round 2 best configuration:', search.best_params_)
    if non_convergence:
        print('CAUTION:', non_convergence, 'convergence warnings; review the CV trials.')

round2_cv_summary = pd.DataFrame(round2_records).set_index('Model')
display(round2_cv_summary.round(5))



ROUND 2: Logistic Regression
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Round 1 mean CV AP: 0.28787
Round 2 mean CV AP: 0.28797
Round 2 best configuration: {'C': 0.1, 'penalty': 'l1', 'solver': 'saga'}

ROUND 2: Histogram Gradient Boosting
Fitting 3 folds for each of 15 candidates, totalling 45 fits
Round 1 mean CV AP: 0.29135
Round 2 mean CV AP: 0.29194
Round 2 best configuration: {'l2_regularization': 30.0, 'learning_rate': 0.05, 'max_iter': 180, 'max_leaf_nodes': 15, 'min_samples_leaf': 50}


,Round 1 CV Average Precision,Round 2 CV Average Precision,CV AP difference,Round 2 CV AP std,Round 2 CV ROC-AUC,Round 2 CV F1 @0.50,CV candidates,Search time (s),Refit time (s),Convergence warnings,Best parameters
Model,,,,,,,,,,,
Logistic Regression,0.28787,0.28797,0.00010,0.00044,0.69466,0.35701,12,3124.67836,202.57846,0,"{""C"": 0.1, ""penalty"": ""l1"", ""solver"": ""saga""}"
Histogram Gradient Boosting,0.29135,0.29194,0.00059,0.00051,0.69761,0.35621,15,630.90575,18.86223,0,"{""l2_regularization"": 30.0, ""learning_rate"": 0..."


## 4. Evaluate on validation


In [4]:
# Part E, Round 2 — 4. Compare the tuned configurations on the untouched validation split.
# Decisions are based on training CV Average Precision; validation is a holdout check.
def r2_metrics(y, probabilities, threshold=0.5):
    predictions = (np.asarray(probabilities) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, predictions, labels=[0, 1]).ravel()
    return {
        'Accuracy': accuracy_score(y, predictions),
        'Precision': precision_score(y, predictions, zero_division=0),
        'Recall': recall_score(y, predictions, zero_division=0),
        'F1-score': f1_score(y, predictions, zero_division=0),
        'F2-score': fbeta_score(y, predictions, beta=2, zero_division=0),
        'ROC-AUC': roc_auc_score(y, probabilities),
        'Average Precision': average_precision_score(y, probabilities),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
    }

val_rows = []
val_preds = pd.DataFrame({'actual': y_val_r2.to_numpy()})
for model_name, model in round2_best_models.items():
    probabilities = model.predict_proba(X_val_r2)[:, 1]
    val_preds[f'{model_name}_prob'] = probabilities
    val_preds[f'{model_name}_pred'] = (probabilities >= 0.5).astype(int)
    val_rows.append({'Model': model_name, **r2_metrics(y_val_r2, probabilities)})

round2_metrics = pd.DataFrame(val_rows).set_index('Model')
round2_metrics.reset_index().to_csv(ROUND2 / 'round2_validation_metrics.csv', index=False)
val_preds.to_csv(ROUND2 / 'round2_validation_predictions.csv', index=False)

compare_rows = []
for model_name in round2_metrics.index:
    for met in ['Precision', 'Recall', 'F1-score', 'ROC-AUC', 'Average Precision']:
        old = float(stage1_metrics_r2.loc[model_name, met])
        new = float(round2_metrics.loc[model_name, met])
        compare_rows.append({'Model': model_name, 'Metric': met,
                             'Round 1 @0.50': old, 'Round 2 @0.50': new,
                             'Difference': new - old})
comparison = pd.DataFrame(compare_rows)
comparison.to_csv(ROUND2 / 'round1_vs_round2_validation.csv', index=False)
print('Round 2 validation metrics at 0.50:')
display(round2_metrics.round(4))
print('Round 2 minus Round 1 on validation:')
display(comparison.pivot(index='Model', columns='Metric',
                         values='Difference').round(5))


Round 2 validation metrics at 0.50:


,Accuracy,Precision,Recall,F1-score,F2-score,ROC-AUC,Average Precision,TN,FP,FN,TP
Model,,,,,,,,,,,
Logistic Regression,0.6335,0.2458,0.6574,0.3578,0.4925,0.6946,0.282,24072,14193,2411,4626
Histogram Gradient Boosting,0.6269,0.2438,0.6672,0.3572,0.4952,0.6972,0.286,23706,14559,2342,4695


Round 2 minus Round 1 on validation:


Metric,Average Precision,F1-score,Precision,ROC-AUC,Recall
Model,,,,,
Histogram Gradient Boosting,-0.00009,-0.00001,0.00014,-0.00045,-0.00114
Logistic Regression,0.00009,0.00006,0.00004,0.00010,0.00014


## 5. Save final Round 2 outputs


In [5]:
# Part E, Round 2 — 5. Save reproducible configurations and a concise interpretation.
config_r2 = {}
for model_name, search in round2_searches_fitted.items():
    config_r2[model_name] = {
        'estimator_class': type(search.best_estimator_).__name__,
        'round2_best_parameters': search.best_params_,
        'class_weight_policy': 'computed from full training labels as in Part D',
        'cv_folds': CV_FOLDS,
        'cv_shuffle': True,
        'seed': SEED,
        'cv_selection_score': 'average_precision',
        'cv_mean_average_precision': float(search.best_score_),
        'default_decision_threshold': 0.50,
        'test_set_accessed': False,
    }
    if model_name == 'Histogram Gradient Boosting':
        config_r2[model_name]['refitted_actual_iterations'] = int(search.best_estimator_.n_iter_)

(ROUND2 / 'round2_model_configurations.json').write_text(
    json.dumps(config_r2, indent=2))

report = [
    '# Part E — Focused Round 2 summary', '',
    '- Objective: refine Logistic Regression and Histogram Gradient Boosting.',
    '- Search selection: mean Average Precision under 3-fold stratified CV on training data.',
    '- Data, target, class weights and threshold 0.50 match the Part D/E comparisons.',
    '- Test set not accessed. Round 1 exports were not modified.', '',
]
for name in round2_cv_summary.index:
    cv_row = round2_cv_summary.loc[name]
    report.extend([
        f'## {name}',
        f'- Round 1 best CV AP: {cv_row["Round 1 CV Average Precision"]:.6f}',
        f'- Round 2 best CV AP: {cv_row["Round 2 CV Average Precision"]:.6f}',
        f'- CV AP difference: {cv_row["CV AP difference"]:+.6f}',
        f'- Best Round 2 parameters: `{cv_row["Best parameters"]}`',
        f'- Validation AP, Round 1 → Round 2: '
        f'{stage1_metrics_r2.loc[name, "Average Precision"]:.6f} → '
        f'{round2_metrics.loc[name, "Average Precision"]:.6f}',
        f'- Validation F1 @0.50, Round 1 → Round 2: '
        f'{stage1_metrics_r2.loc[name, "F1-score"]:.6f} → '
        f'{round2_metrics.loc[name, "F1-score"]:.6f}',
        '',
    ])
report += [
    '## Interpretation guidance',
    '- Prefer CV AP for hyperparameter selection; do not repeatedly select models on validation metrics.',
    '- A marginal CV AP change is weak evidence of general improvement.',
    '- Changing regularisation may leave Logistic Regression effectively unchanged.',
    '- HGB early stopping may make actual boosting iterations lower than max_iter.',
    '- Report both improvements and declines. Freeze model configurations before Part G.',
]
(ROUND2 / 'round2_report_summary.md').write_text('\n'.join(report))
print('Round 2 completed. Files saved under:', ROUND2)
for path in sorted(ROUND2.iterdir()):
    print('-', path.name)


Round 2 completed. Files saved under: /content/drive/MyDrive/ML_Group/Part_E_Results/Round_2
- histogram_gradient_boosting_cv_trials.csv
- logistic_regression_cv_trials.csv
- round1_vs_round2_validation.csv
- round2_best_parameters.csv
- round2_model_configurations.json
- round2_report_summary.md
- round2_validation_metrics.csv
- round2_validation_predictions.csv
